# 06a — Store Agent Traces in Unity Catalog

**Jackson and Jackson HR Digital**

This notebook moves the Hire Right agent's MLflow traces out of experiment storage and into
**Unity Catalog Delta tables** (OpenTelemetry format), so they can be governed with UC
permissions, queried with SQL, and analysed with Genie and AI/BI dashboards.

### What this notebook does
1. Creates a new MLflow experiment **linked to a UC trace location** (`<catalog>.<schema>.<table_prefix>_otel_*`)
2. Grants explicit `MODIFY` + `SELECT` on the trace tables (required — `ALL_PRIVILEGES` is not sufficient)
3. Migrates historical traces (spans, assessments, tags) from the legacy experiment
4. Verifies row counts and prints the trace table schemas
5. Returns the UC experiment ID so notebook `06` logs and deploys the agent against it

### Why a new experiment?
An existing experiment cannot be re-bound to a UC trace location, so the legacy
`agent-predictive-hiring-evals` experiment is left untouched and its traces are **copied**.

### Prerequisites
- The target schema exists (e.g. `bldemos.hrd_2030_agent_observability`)
- A SQL warehouse the running user can `CAN USE`
- Serverless or DBR 15.3+ (required by the trace migration job)

**Next notebook:** `06_evaluate_register_agent.ipynb`

## Setup

Install MLflow 3.14+ (UC trace storage) and `databricks-agents` 1.10.1+ (trace migration).

In [ ]:
%pip install -q "mlflow[databricks]==3.17.1" "databricks-agents>=1.10.1" python-dotenv

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("trace_catalog",     "bldemos",                        "UC Catalog (traces)")
dbutils.widgets.text("trace_schema",      "hrd_2030_agent_observability",   "UC Schema (traces)")
dbutils.widgets.text("trace_table_prefix", "hrd_agent",                    "Trace Table Prefix")
dbutils.widgets.text("uc_experiment",     "hrd-agent-traces",     "UC-linked MLflow Experiment")
dbutils.widgets.text("source_experiment", "agent-predictive-hiring-evals",  "Legacy Experiment (migrate from)")
dbutils.widgets.text("warehouse_id",      "",                               "SQL Warehouse ID")
dbutils.widgets.dropdown("migrate_traces", "true", ["true", "false"],       "Migrate Legacy Traces")

In [ ]:
import os
from dotenv import load_dotenv

# Locate the project root from the running notebook's workspace path and load .env
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
load_dotenv(f"{project_root}/.env")

# Widget takes priority (job passes it); fall back to .env for interactive runs
trace_catalog      = dbutils.widgets.get("trace_catalog")      or os.getenv("TRACE_CATALOG", "bldemos")
trace_schema       = dbutils.widgets.get("trace_schema")       or os.getenv("TRACE_SCHEMA", "hrd_2030_agent_observability")
trace_table_prefix = dbutils.widgets.get("trace_table_prefix") or os.getenv("TRACE_TABLE_PREFIX", "hrd_agent")
uc_experiment      = dbutils.widgets.get("uc_experiment")      or os.getenv("MLFLOW_EXPERIMENT_AGENT_UC", "hrd-agent-traces")
source_experiment  = dbutils.widgets.get("source_experiment")  or os.getenv("MLFLOW_EXPERIMENT_AGENT", "agent-predictive-hiring-evals")
warehouse_id       = dbutils.widgets.get("warehouse_id")       or os.getenv("DATABRICKS_WAREHOUSE_ID", "")
migrate_traces     = dbutils.widgets.get("migrate_traces").lower() == "true"

if not warehouse_id:
    raise ValueError("warehouse_id is required — UC trace tables are provisioned and queried through a SQL warehouse.")

# MLflow uses this warehouse to provision the trace tables and to read UC traces in the UI/SDK
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = warehouse_id

_db_username      = spark.sql("SELECT current_user()").collect()[0][0]
uc_experiment_path = f"/Users/{_db_username}/{uc_experiment}"
source_experiment_path = f"/Users/{_db_username}/{source_experiment}"
trace_table_base  = f"{trace_catalog}.{trace_schema}.{trace_table_prefix}"

print(f"Trace location       : {trace_table_base}_otel_*")
print(f"UC experiment        : {uc_experiment_path}")
print(f"Legacy experiment    : {source_experiment_path}")
print(f"SQL warehouse        : {warehouse_id}")
print(f"Migrate traces       : {migrate_traces}")

## Create the UC-Linked Experiment

`mlflow.set_experiment(..., trace_location=UnityCatalog(...))` creates the experiment (if needed),
provisions the four OTel Delta tables, and binds them to the experiment. Re-running is safe:
an experiment already bound to the same location is reused; one bound elsewhere raises.

In [ ]:
import mlflow
from mlflow.entities.trace_location import UnityCatalog

uc_location = UnityCatalog(
    catalog_name=trace_catalog,
    schema_name=trace_schema,
    table_prefix=trace_table_prefix,
)

experiment = mlflow.set_experiment(experiment_name=uc_experiment_path, trace_location=uc_location)
uc_experiment_id = experiment.experiment_id

assert isinstance(experiment.trace_location, UnityCatalog), \
    f"Experiment is not linked to a UC trace location: {experiment.trace_location}"

print(f"✓ Experiment      : {experiment.name}")
print(f"  Experiment ID   : {uc_experiment_id}")
print(f"  Trace location  : {experiment.trace_location}")

## Grant Trace Table Permissions

Writers and readers of UC traces need `USE CATALOG`, `USE SCHEMA`, and explicit `MODIFY` + `SELECT`
on each trace table. The serving endpoint writes traces as the identity behind its `DATABRICKS_TOKEN`
(the deploying user), so a direct grant to that user covers both notebook and endpoint traces.

In [ ]:
trace_tables = [f"{trace_table_base}_otel_{t}" for t in ("spans", "logs", "metrics", "annotations")]

spark.sql(f"GRANT USE CATALOG ON CATALOG {trace_catalog} TO `{_db_username}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {trace_catalog}.{trace_schema} TO `{_db_username}`")
for _t in trace_tables:
    spark.sql(f"GRANT MODIFY, SELECT ON TABLE {_t} TO `{_db_username}`")
    print(f"✓ MODIFY, SELECT on {_t} → {_db_username}")

## Migrate Legacy Traces

Copies traces, spans, assessments (LLM judge scores), tags, and metadata from the legacy experiment
into the UC trace tables. The source experiment's traces are not modified, and the job bookmarks
its progress, so re-runs only pick up traces logged since the last run.

In [ ]:
from databricks.migrations.migrate_traces_to_uc import run as migrate_traces_to_uc

source_exp = mlflow.get_experiment_by_name(source_experiment_path)

if not migrate_traces:
    print("Skipping migration (migrate_traces=false).")
elif source_exp is None:
    print(f"⚠️  Legacy experiment not found: {source_experiment_path} — nothing to migrate.")
else:
    print(f"Migrating traces {source_exp.experiment_id} → {uc_experiment_id} ...")
    migrate_traces_to_uc(
        source_experiment_id=source_exp.experiment_id,
        target_experiment_id=uc_experiment_id,
    )
    print("✓ Migration complete")

## Verify Trace Tables

Confirm the tables are populated and preview the MLflow-managed `<prefix>_trace_unified` view (the base for the analytics views in notebook `11`).

In [ ]:
# Trace tables contain VARIANT columns, so select only scalar columns here —
# older serverless Spark Connect clients cannot deserialize VARIANT schemas.
for _t in trace_tables:
    _n = spark.sql(f"SELECT COUNT(*) AS n FROM {_t}").collect()[0]["n"]
    print(f"{_t:<70} {_n:>8,} rows")

print()
spark.sql(f"DESCRIBE TABLE {trace_table_base}_otel_spans").show(100, truncate=False)

In [ ]:
# One row per trace from the MLflow-managed <prefix>_trace_unified view
display(spark.sql(f"""
    SELECT trace_id, request_time, state,
           ROUND(execution_duration_ms / 1000, 2) AS latency_seconds,
           SIZE(spans)                             AS span_count,
           SIZE(assessments)                       AS assessment_count
    FROM {trace_table_base}_trace_unified
    ORDER BY request_time DESC
    LIMIT 10
"""))

## Summary

| Item | Value |
|---|---|
| UC experiment | `/Users/<you>/hrd-agent-traces` |
| Trace tables | `<catalog>.<schema>.hrd_agent_otel_{spans,logs,metrics,annotations}` |
| Legacy experiment | `agent-predictive-hiring-evals` (unchanged) |

Notebook `06` now sets this experiment as active before logging the model, so `agents.deploy()`
points the serving endpoint's `MLFLOW_EXPERIMENT_ID` at it and production traces land in UC.

In [ ]:
# Return the UC experiment ID so run_all.ipynb / the job can pass it to notebook 06
dbutils.jobs.taskValues.set(key="uc_experiment_id", value=uc_experiment_id)
dbutils.notebook.exit(uc_experiment_id)